# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides guidance for loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema, accessible via a URL.

**Citation:**
"Mugotitsa, B., Maina, D., Owoko, H., Akinyi, L., Greenfield, J., Todd, J., Kavu, T., and Kiragga, A. 2026. A FAIR2 Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa. Frontiers."

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset's Croissant schema using `mlcroissant` and inspect the metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Print the dataset's basic metadata
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

print("\nOther metadata fields:")
pprint.pprint({
    'Date Published': dataset.metadata.datePublished,
    'Identifier': dataset.metadata.identifier,
    'Keywords': dataset.metadata.keywords,
    'License': dataset.metadata.license
})

## 2. Data Overview
List all available record sets in the dataset, along with their fields and their `@id` values.

In Croissant, data is organized into `record sets`, each with one or more fields (columns). Let's enumerate them by `@id`.

In [ ]:
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in the metadata.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs['@id']} - {rs.get('name', 'No name')}")
        print("  Fields:")
        for field in rs['field']:
            if isinstance(field, dict):
                field_id = field.get('@id', None)
                field_name = field.get('name', None)
            else:
                field_id = field
                field_name = None
            print(f"    - Field @id: {field_id} Name: {field_name}")
        print()
        if 'column' in rs:
            print("  Columns:")
            for col in rs['column']:
                col_id = col.get('@id', col) if isinstance(col, dict) else col
                print(f"    - Column @id: {col_id}")
else:
    # If record sets are not populated directly in the metadata, try listing them using the loader
    print("No record sets explicitly defined in the schema metadata. Attempting to infer possible data sources...")
    try:
        # Find possible data files
        if hasattr(dataset.metadata, 'distribution'):
            for d in dataset.metadata.distribution:
                print(f"Distribution: {d.get('@id', d)}")
    except Exception as e:
        print(f"Exception when accessing distributions: {e}")

## 3. Data Extraction
Let's extract available data by specifying the record set `@id`. Since not all datasets strictly populate the `recordSet` array, we can attempt to infer possible record set IDs from the schema. For this dataset, we'll inspect which record sets are available and attempt loading them.

**Note:** In Croissant, `@id` values are used for all references. For demonstration, we will attempt to load data using a common table record set ID, falling back to the file distribution `@id`.

In [ ]:
from mlcroissant import utils

dataframes = {}
# Step 1: Try to get record sets from the metadata's recordSet list
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    record_sets = dataset.metadata.recordSet
    # Get the list of record set @ids
    record_set_ids = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in record_sets]
    print("Found record sets:", record_set_ids)
else:
    # If not found in recordSet, fall back to distributions (usually CSV files)
    if hasattr(dataset.metadata, 'distribution'):
        # Some distributions may be documentation; filter by encodingFormat if possible
        record_set_ids = [d['@id'] if isinstance(d, dict) and '@id' in d else d for d in dataset.metadata.distribution]
        print("Falling back to distributions as possible record sets:", record_set_ids)
    else:
        record_set_ids = []

# Load each record set as available
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for record set @id: {record_set_id} (shape: {dataframes[record_set_id].shape})")
        else:
            print(f"No records found for record set @id: {record_set_id}")
    except Exception as e:
        print(f"Error loading records for record set @id: {record_set_id}: {e}")

# Pick the largest or first DataFrame as an example
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    print(f"\nAvailable columns for record set {example_record_set_id}:")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No tabular data extracted.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (e.g., from GAD-7, PHQ-9, or PSQ scores if present) in the main record set, filter, normalize, and group by a demographic category.

We will reference fields and columns by their Croissant `@id` (or column name if not present). Adapt the field IDs below as found in the data extraction step.

**Example:** For demonstration, we'll use the `gad7_total_score` field and group by `gender` if those exist. Replace with the proper field/column `@id` as needed.

In [ ]:
# Specify the record set and fields by their @id (from previous step)
record_set_id = example_record_set_id  # use the loaded record set @id

# Try common mental health score fields - adapt to actual @id/column name if not found
possible_numeric_fields = ['gad7_total_score', 'phq9_total_score', 'psq_score', 'GAD7_TOTAL', 'PHQ9_TOTAL', 'PSQ']
numeric_field = None
for col in dataframes[record_set_id].columns:
    for pf in possible_numeric_fields:
        if pf.lower() == col.lower():
            numeric_field = col
            break
    if numeric_field:
        break
if not numeric_field:
    # Fallback - find first numeric column
    for col in dataframes[record_set_id].select_dtypes(include='number').columns:
        numeric_field = col
        break

print(f"Using numeric field: {numeric_field}")

# Use a common group field: gender, age_group or village
possible_group_fields = ['gender', 'Gender', 'village', 'Village', 'age_group', 'AgeGroup', 'marital_status', 'level_of_education']
group_field = None
for col in dataframes[record_set_id].columns:
    for pf in possible_group_fields:
        if pf.lower() == col.lower():
            group_field = col
            break
    if group_field:
        break

df = dataframes[record_set_id]

if numeric_field is not None:
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Records with {numeric_field} > {threshold}")
    display(filtered_df.head())

    # Normalize the selected numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Top 5 rows after normalizing '{numeric_field}':")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Group by group_field if available
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} grouped by {group_field}:")
        display(grouped_df)
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Let's create simple plots showing the distribution of the selected mental health score, and its average by a group field such as gender.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the selected numeric field
if numeric_field is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(9,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("Cannot plot numeric field - not found.")

## 6. Conclusion
In this notebook, we explored the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using mlcroissant by:
- Loading and inspecting Croissant metadata
- Enumerating record sets and fields by their Croissant `@id`
- Loading tabular data and extracting main survey records
- Filtering and normalizing mental health assessment scores (e.g., GAD-7 or PHQ-9) by their column `@id`
- Grouping and visualizing data by demographic attributes

The Croissant standard and `mlcroissant` help ensure functional, reusable, and reproducible FAIR (Findable, Accessible, Interoperable, Reusable) data workflows.

Further analysis could include predictive modeling for mental health risk factors, or integration with other surveys following the Croissant structure.